### Inferência Causal

*PSI e mercado de trabalho formal*


Regressão DiD


**Pergunta:** municípios mais expostos ao Finame *antes* do PSI cresceram mais (emprego e massa salarial) *depois* do programa?

### Pacotes & Imports

In [1]:
%pip install pandas numpy scipy statsmodels linearmodels matplotlib seaborn scikit-learn openpyxl Unidecode

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# --- Manipulação de dados ---
import numpy as np
import pandas as pd

# --- Estatística e testes de hipótese ---
from scipy import stats                      # testes t, qui-quadrado, normalidade, etc.

# --- Regressões (MQO, DiD, efeitos fixos via fórmula) ---
import statsmodels.api as sm                 # sm.OLS, sm.Logit, sm.add_constant
import statsmodels.formula.api as smf        # smf.ols('y ~ tratado * pos + C(ano)', data=df)
from statsmodels.stats.diagnostic import het_breuschpagan   # heterocedasticidade
from statsmodels.stats.outliers_influence import variance_inflation_factor  # multicolinearidade
from statsmodels.iolib.summary2 import summary_col           # tabela comparando modelos

# --- Dados em painel (efeitos fixos/aleatórios, TWFE para DiD) ---
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
from linearmodels.panel import compare as compare_panel

# --- Pareamento / escore de propensão (opcional, PSM + DiD) ---
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors

# --- Gráficos (tendências paralelas, event study) ---
import matplotlib.pyplot as plt
import seaborn as sns

# --- Configurações ---
import warnings
warnings.filterwarnings('ignore')
pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')

## --- outros 
from datetime import datetime

from unidecode import unidecode

In [3]:
data_hora = datetime.now().strftime("%d-%m-%Y_%H-%M")

### Bases para o painel



O objetivo central é estimar esses efeitos explorando diferenças predeterminadas na exposição municipal ao Finame antes de 2009. Os objetivos específicos são: 
1. construir um painel município-ano para 2005-2013, combinando desembolsos do BNDES/Finame, indicadores da RAIS e população do IBGE; 
2. construir uma medida de exposição pré-PSI com os desembolsos do Finame de 2002-2004, anteriores ao período de análise;  
3. estimar a mudança diferencial pós-PSI segundo a exposição inicial; 
4. estimar um estudo de evento para examinar pré-tendências e a dinâmica dos efeitos; 
5. testar se o efeito cresce com o nível de exposição, por meio de uma especificação por faixas de exposição.


### df_finame

In [4]:
df_finame = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\desembolsos-mensais_Finame.xlsx", sheet_name='sheet_1')

In [5]:
df_finame

,ano,mes,forma_de_apoio,produto,instrumento_financeiro,inovacao,porte_de_empresa,regiao,uf,municipio,municipio_codigo,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,desembolsos_reais
0,1995.0000,1.0000,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,CENTRO OESTE,MATO GROSSO,BARRA DO BUGRES,5101704.0000,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,4521720.3700
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1995.0000,1.0000,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,NORTE,AMAZONAS,NOVO AIRAO,1303205.0000,COMﾉRCIO E SERVIﾇOS,TRANSPORTE AQUAVIﾁRIO,INFRAESTRUTURA,OUTROS TRANSPORTES,2099499.9900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367837,2026.0000,3.0000,INDIRETA,BNDES FINAME,MODERFROTA,NﾃO,PEQUENA,SUL,PARANA,JURANDA,4112959.0000,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203480.0000
367838,2026.0000,3.0000,INDIRETA,BNDES FINAME,MATERIAIS INDUSTRIALIZADOS,NﾃO,MﾉDIA,SUDESTE,ESPIRITO SANTO,SERRA,3205002.0000,COMﾉRCIO E SERVIﾇOS,COMﾉRCIO,COMﾉRCIO E SERVIﾇOS,COMﾉRCIO E SERVIﾇOS,203330.6400
367839,2026.0000,3.0000,INDIRETA,BNDES FINAME,PRONAF INVESTIMENTO,NﾃO,PEQUENA,SUL,SANTA CATARINA,ROMELANDIA,4215208.0000,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203202.0000
367840,2026.0000,3.0000,INDIRETA,BNDES FINAME,PRONAF INVESTIMENTO,NﾃO,PEQUENA,SUDESTE,ESPIRITO SANTO,COLATINA,3201506.0000,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203000.0000


Células vazias e entrega de números inteiros em formato de decimais

Limpar a coluna


contar o que está vazio

In [6]:
df_finame = df_finame.dropna(subset=["municipio_codigo"])

df_finame[["ano", "mes", "municipio_codigo"]].isna().sum()

ano                 0
mes                 0
municipio_codigo    0
dtype: int64

In [7]:
for col in ["ano", "mes", "municipio_codigo"]:
    df_finame[col] = df_finame[col].astype(int)

In [8]:
siglas = {
    "ACRE": "AC", "ALAGOAS": "AL", "AMAPA": "AP",
    "AMAZONAS": "AM", "BAHIA": "BA", "CEARA": "CE",
    "DISTRITO FEDERAL": "DF", "ESPIRITO SANTO": "ES",
    "GOIAS": "GO", "MARANHAO": "MA", "MATO GROSSO": "MT",
    "MATO GROSSO DO SUL": "MS", "MINAS GERAIS": "MG",
    "PARA": "PA", "PARAIBA": "PB", "PARANA": "PR",
    "PERNAMBUCO": "PE", "PIAUI": "PI",
    "RIO DE JANEIRO": "RJ", "RIO GRANDE DO NORTE": "RN",
    "RIO GRANDE DO SUL": "RS", "RONDONIA": "RO",
    "RORAIMA": "RR", "SANTA CATARINA": "SC",
    "SAO PAULO": "SP", "SERGIPE": "SE", "TOCANTINS": "TO"
}

df_finame["uf_sigla"] = (
    df_finame["uf"].map(siglas)
)

In [9]:
df_finame

,ano,mes,forma_de_apoio,produto,instrumento_financeiro,inovacao,porte_de_empresa,regiao,uf,municipio,municipio_codigo,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,desembolsos_reais,uf_sigla
0,1995,1,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,CENTRO OESTE,MATO GROSSO,BARRA DO BUGRES,5101704,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,4521720.3700,MT
4,1995,1,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,NORTE,AMAZONAS,NOVO AIRAO,1303205,COMﾉRCIO E SERVIﾇOS,TRANSPORTE AQUAVIﾁRIO,INFRAESTRUTURA,OUTROS TRANSPORTES,2099499.9900,AM
5,1995,1,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,CENTRO OESTE,MATO GROSSO DO SUL,DOURADOS,5003702,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,2064969.2400,MS
6,1995,1,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,SUDESTE,SAO PAULO,MOGI DAS CRUZES,3530607,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,2058083.2900,SP
7,1995,1,INDIRETA,BNDES FINAME,NaN,NaN,GRANDE,SUDESTE,SAO PAULO,SAO PAULO,3550308,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,1892247.4100,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
367837,2026,3,INDIRETA,BNDES FINAME,MODERFROTA,NﾃO,PEQUENA,SUL,PARANA,JURANDA,4112959,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203480.0000,PR
367838,2026,3,INDIRETA,BNDES FINAME,MATERIAIS INDUSTRIALIZADOS,NﾃO,MﾉDIA,SUDESTE,ESPIRITO SANTO,SERRA,3205002,COMﾉRCIO E SERVIﾇOS,COMﾉRCIO,COMﾉRCIO E SERVIﾇOS,COMﾉRCIO E SERVIﾇOS,203330.6400,ES
367839,2026,3,INDIRETA,BNDES FINAME,PRONAF INVESTIMENTO,NﾃO,PEQUENA,SUL,SANTA CATARINA,ROMELANDIA,4215208,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203202.0000,SC
367840,2026,3,INDIRETA,BNDES FINAME,PRONAF INVESTIMENTO,NﾃO,PEQUENA,SUDESTE,ESPIRITO SANTO,COLATINA,3201506,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,AGROPECUﾁRIA,203000.0000,ES


### df_PSI

In [10]:
df_PSI = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\desembolsos-mensais_PSI.xlsx", sheet_name='desembolsos-mensais_PSI')

df_PSI = df_PSI[df_PSI['ano'].isin([2009, 2010, 2011, 2012, 2013])]

In [11]:
df_PSI

,ano,mes,forma_de_apoio,produto,instrumento_financeiro,inovacao,porte_de_empresa,regiao,uf,municipio,municipio_codigo,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,desembolsos_reais
0,2009,8,DIRETA,BNDES FINEM,PSI - BK - Demais Itens,N鬃,GRANDE,NORTE,RONDONIA,PORTO VELHO,1100205,IND湒TRIA DE TRANSFORMAO,MINERAL N鬃 MET簇ICO,INDUSTRIA,OUTRAS,17000000.0000
1,2009,8,DIRETA,BNDES FINEM,PSI - BK - Demais Itens,N鬃,GRANDE,CENTRO OESTE,MATO GROSSO,NOBRES,5105903,IND湒TRIA DE TRANSFORMAO,MINERAL N鬃 MET簇ICO,INDUSTRIA,OUTRAS,8000000.0000
2,2009,8,INDIRETA,BNDES FINAME,PSI - BK - Demais Itens,N鬃,GRANDE,SUDESTE,RIO DE JANEIRO,RIO DE JANEIRO,3304557,COM仂CIO E SERVIS,CONSTRUO,COM仂CIO E SERVIS,COM仂CIO E SERVIS,5061388.7500
3,2009,8,INDIRETA,BNDES FINAME,PSI - BK - Demais Itens,N鬃,GRANDE,SUL,PARANA,CURITIBA,4106902,COM仂CIO E SERVIS,CONSTRUO,COM仂CIO E SERVIS,COM仂CIO E SERVIS,2501578.0000
4,2009,8,INDIRETA,BNDES FINAME,PSI - BK - Demais Itens,N鬃,MICRO,CENTRO OESTE,GOIAS,RIO VERDE,5218805,AGROPECU糠IA,AGROPECU糠IA,AGROPECU糠IA,AGROPECU糠IA,2429100.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
189375,2013,9,DIRETA,BNDES FINEM,PSI - Projetos Transformadores,SIM,GRANDE,SUDESTE,MINAS GERAIS,RIBEIRAO DAS NEVES,3154606,IND湒TRIA DE TRANSFORMAO,"EQUIP INFO, ELETRONICO, 紘ICO",INDUSTRIA,MEC齋ICA,20429555.0100
189376,2013,9,INDIRETA,BNDES FINEM,PSI - Projetos Transformadores,SIM,GRANDE,SUDESTE,MINAS GERAIS,RIBEIRAO DAS NEVES,3154606,IND湒TRIA DE TRANSFORMAO,"EQUIP INFO, ELETRONICO, 紘ICO",INDUSTRIA,MEC齋ICA,6566642.6800
189377,2013,10,DIRETA,BNDES FINEM,PSI - Projetos Transformadores,SIM,GRANDE,SUDESTE,MINAS GERAIS,RIBEIRAO DAS NEVES,3154606,IND湒TRIA DE TRANSFORMAO,"EQUIP INFO, ELETRONICO, 紘ICO",INDUSTRIA,MEC齋ICA,6764000.0000
189378,2013,10,INDIRETA,BNDES FINEM,PSI - Projetos Transformadores,SIM,GRANDE,SUDESTE,MINAS GERAIS,RIBEIRAO DAS NEVES,3154606,IND湒TRIA DE TRANSFORMAO,"EQUIP INFO, ELETRONICO, 紘ICO",INDUSTRIA,MEC齋ICA,2174000.0000


### df_ipca

In [12]:
df_ipca = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\IPCA_STP-20261019203512642.xlsx", sheet_name='IPCA')

df_ipca

,Data,IPCA - Var. % mensal
0,2000-01-01,0.6200
1,2000-02-01,0.1300
2,2000-03-01,0.2200
3,2000-04-01,0.4200
4,2000-05-01,0.0100
...,...,...
175,2014-08-01,0.2500
176,2014-09-01,0.5700
177,2014-10-01,0.4200
178,2014-11-01,0.5100


### df_ibge_pib_municipal

In [13]:
df_pib_munic_2002_2009 = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\PIB dos Municípios - base de dados 2002-2009.xlsx", sheet_name='PIB_dos_Municípios')

df_pib_munic_2002_2009

,Ano,Código da Grande Região,Nome da Grande Região,Código da Unidade da Federação,UF,NOME UF,Código do Município,Nome do Município,Código da Região Rural,Nome da Região Rural,Região rural (segundo classificação do núcleo),Amazônia Legal,Semiárido,"Valor adicionado bruto da Agropecuária, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)","Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)","Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)","Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)"
0,2002,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,27013.2230,9376.8710,24651.1130,42700.5230,103741.7290,7549.2660,111290.9950,4047.8300
1,2002,1,Norte,11,RO,Rondônia,1100023,Ariquemes,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,28590.1780,99922.9360,161851.5490,116562.1820,406926.8450,42665.9720,449592.8160,5667.3700
2,2002,1,Norte,11,RO,Rondônia,1100031,Cabixi,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,10883.8010,1961.9070,5128.8760,11925.0010,29899.5850,1867.9350,31767.5200,4246.9900
3,2002,1,Norte,11,RO,Rondônia,1100049,Cacoal,5105,Região Rural do Centro Sub-regional de Vilhena...,Região Rural de Centro Sub-regional,Sim,Não,32810.3760,98292.7640,175748.7030,117968.3030,424820.1460,49622.9510,474443.0970,6353.2700
4,2002,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,11283.7930,6651.4750,25306.0090,27731.3660,70972.6430,8200.9710,79173.6140,4442.4700
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44496,2009,5,Centro-oeste,52,GO,Goiás,5222005,Vianópolis,5204,Região Rural da Metrópole Nacional de Brasília,Região Rural de Metrópole Nacional,Não,Não,53529.0850,14813.1680,58617.5850,26216.1580,153175.9960,11160.3410,164336.3370,12807.7600
44497,2009,5,Centro-oeste,52,GO,Goiás,5222054,Vicentinópolis,5206,Região Rural do Centro Sub-regional de Itumbiara,Região Rural de Centro Sub-regional,Não,Não,63465.0400,11101.4180,38070.9060,13974.0180,126611.3820,6819.0300,133430.4130,21898.9700
44498,2009,5,Centro-oeste,52,GO,Goiás,5222203,Vila Boa,5203,Região Rural do Centro de Zona de Campos Belos,Região Rural de Centro de zona,Não,Não,25391.5200,12652.8350,19921.0140,11463.2990,69428.6690,3938.7130,73367.3810,16026.0800
44499,2009,5,Centro-oeste,52,GO,Goiás,5222302,Vila Propício,5202,Região Rural do Centro Sub-regional de Anápolis,Região Rural de Centro Sub-regional,Não,Não,49814.2180,26168.4280,16793.2110,11847.7400,104623.5970,2501.7940,107125.3910,19989.8100


In [14]:
df_pib_munic_2010_2023 = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\PIB dos Municípios - base de dados 2010-2023.xlsx", sheet_name='PIB dos Municípios')

df_pib_munic_2010_2023

,Ano,Código da Grande Região,Nome da Grande Região,Código da Unidade da Federação,Sigla da Unidade da Federação,Nome da Unidade da Federação,Código do Município,Nome do Município,Região Metropolitana,Código da Mesorregião,Nome da Mesorregião,Código da Microrregião,Nome da Microrregião,Código da Região Geográfica Imediata,Nome da Região Geográfica Imediata,Município da Região Geográfica Imediata,Código da Região Geográfica Intermediária,Nome da Região Geográfica Intermediária,Município da Região Geográfica Intermediária,Código Concentração Urbana,Nome Concentração Urbana,Tipo Concentração Urbana,Código Arranjo Populacional,Nome Arranjo Populacional,Hierarquia Urbana,Hierarquia Urbana (principais categorias),Código da Região Rural,Nome da Região Rural,Região rural (segundo classificação do núcleo),Amazônia Legal,Semiárido,Cidade-Região de São Paulo,"Valor adicionado bruto da Agropecuária, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)","Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)","Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)","Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)",Atividade com maior valor adicionado bruto,Atividade com segundo maior valor adicionado bruto,Atividade com terceiro maior valor adicionado bruto
0,2010,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,NaN,1102,Leste Rondoniense,11006,Cacoal,110005,Cacoal,do Entorno,1102,Ji-Paraná,do Entorno,NaN,NaN,NaN,NaN,NaN,Centro Local,Centro Local,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,Não,69260.3910,16118.5340,62496.1850,93244.6560,241119.7670,20957.1110,262076.8780,10731.1800,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
1,2010,1,Norte,11,RO,Rondônia,1100023,Ariquemes,NaN,1102,Leste Rondoniense,11003,Ariquemes,110002,Ariquemes,Polo,1101,Porto Velho,do Entorno,NaN,NaN,NaN,NaN,NaN,Centro Sub-Regional A,Centro Sub-Regional,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,Não,73711.6430,287138.5850,494946.2670,343867.7310,1199664.2270,165029.5530,1364693.7800,15103.8600,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
2,2010,1,Norte,11,RO,Rondônia,1100031,Cabixi,NaN,1102,Leste Rondoniense,11008,Colorado do Oeste,110006,Vilhena,do Entorno,1102,Ji-Paraná,do Entorno,NaN,NaN,NaN,NaN,NaN,Centro Local,Centro Local,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não,Não,24300.8220,3252.5060,12677.2100,25170.2350,65400.7720,4210.3420,69611.1140,11033.6200,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
3,2010,1,Norte,11,RO,Rondônia,1100049,Cacoal,NaN,1102,Leste Rondoniense,11006,Cacoal,110005,Cacoal,Polo,1102,Ji-Paraná,do Entorno,NaN,NaN,NaN,NaN,NaN,Capital Regional C,Capital Regional,5105,Região Rural do Centro Sub-regional de Vilhena...,Região Rural de Centro Sub-regional,Sim,Não,Não,95259.2030,182051.5370,465447.3250,298454.3090,1041212.3740,145281.7170,1186494.0910,15095.1500,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
4,2010,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,NaN,1102,Leste Rondoniense,11008,Colorado do Oeste,110006,Vilhena,do Entorno,1102,Ji-Paraná,do Entorno,NaN,NaN,NaN,NaN,NaN,Centro de Zona B,Centro de Zona,1101,Região Rural da Capital Regional de Porto Velho,Região Rural de Capital Regional,Sim,Não

### df_RAIS_emprego_salario

In [15]:
df_rais = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\rais_empr_remuner_2000_2014.xlsx", sheet_name='RAIS')
df_rais

,ano,sigla_uf,id_municipio,subsetor_ibge,emprego_3112,prop_homens,idade_media,prop_mulheres,prop_preta,prop_parda,prop_negra,valor_salario_contratual,valor_remuneracao_media,valor_remuneracao_dezembro
0,2001,IGNORADO,NaN,NaN,39,0.5897,41.2308,0.4103,0.0000,0.0000,0.0000,NaN,807.8531,833.0259
1,2002,IGNORADO,NaN,NaN,54,0.5000,37.8704,0.5000,0.0000,0.0000,0.0000,NaN,802.8307,782.9515
2,2003,IGNORADO,NaN,NaN,2,0.5000,25.5000,0.5000,0.0000,0.0000,0.0000,NaN,451.7350,569.8800
3,2004,IGNORADO,NaN,NaN,1,0.0000,28.0000,1.0000,0.0000,0.0000,0.0000,NaN,403.3200,413.3200
4,2005,IGNORADO,NaN,NaN,5,0.4000,35.4000,0.6000,0.0000,0.0000,0.0000,NaN,1231.6060,652.6220
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83304,2010,DF,5300108.0000,NaN,1099832,0.6369,35.7004,0.3631,0.0242,0.2275,0.2516,NaN,3274.0976,3609.4194
83305,2011,DF,5300108.0000,NaN,1156908,0.6327,35.8587,0.3673,0.0246,0.2364,0.2610,NaN,3417.8013,3695.1621
83306,2012,DF,5300108.0000,NaN,1181649,0.6277,36.1439,0.3723,0.0242,0.2379,0.2621,NaN,3639.5494,3888.0370
83307,2013,DF,5300108.0000,NaN,1302284,0.6339,35.5421,0.3661,0.0223,0.2341,0.2564,NaN,3782.9309,4097.1058


In [16]:
df_rais.columns.tolist()

['ano',
 'sigla_uf',
 'id_municipio',
 'subsetor_ibge',
 'emprego_3112',
 'prop_homens',
 'idade_media',
 'prop_mulheres',
 'prop_preta',
 'prop_parda',
 'prop_negra',
 'valor_salario_contratual',
 'valor_remuneracao_media',
 'valor_remuneracao_dezembro']

In [17]:
df_rais = df_rais.dropna(subset=["id_municipio"])

# rais[["id_municipio", "valor_salario_contratual", "valor_remuneracao_media",  "valor_remuneracao_dezembro"]].isna().sum()

In [18]:
for col in ["id_municipio"]:
    df_rais[col] = df_rais[col].astype(int)

df_rais[col].dtypes


dtype('int64')

In [19]:
# rais = rais[['ano',
#              'sigla_uf',
#              'id_municipio',
#              'emprego_3112',
#              'prop_homens',
#              'idade_media',
#              'prop_mulheres',
#              'prop_preta',
#              'prop_parda',
#              'prop_negra',
#              'valor_remuneracao_media',
#              'valor_remuneracao_dezembro']]

df_rais.reset_index(drop=True, inplace=True)

df_rais

,ano,sigla_uf,id_municipio,subsetor_ibge,emprego_3112,prop_homens,idade_media,prop_mulheres,prop_preta,prop_parda,prop_negra,valor_salario_contratual,valor_remuneracao_media,valor_remuneracao_dezembro
0,2000,RO,1100015,NaN,1196,0.5677,31.1931,0.4323,0.0000,0.0000,0.0000,NaN,287.4081,290.0647
1,2001,RO,1100015,NaN,1185,0.5789,31.6574,0.4211,0.0000,0.0000,0.0000,NaN,330.5743,323.1142
2,2002,RO,1100015,NaN,1440,0.5785,32.2160,0.4215,0.0000,0.0000,0.0000,NaN,438.4699,409.4442
3,2003,RO,1100015,NaN,1629,0.5856,32.4205,0.4144,0.0000,0.0000,0.0000,NaN,473.1530,503.8662
4,2004,RO,1100015,NaN,1610,0.6050,32.2075,0.3950,0.0000,0.0000,0.0000,NaN,520.5894,533.4424
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83297,2010,DF,5300108,NaN,1099832,0.6369,35.7004,0.3631,0.0242,0.2275,0.2516,NaN,3274.0976,3609.4194
83298,2011,DF,5300108,NaN,1156908,0.6327,35.8587,0.3673,0.0246,0.2364,0.2610,NaN,3417.8013,3695.1621
83299,2012,DF,5300108,NaN,1181649,0.6277,36.1439,0.3723,0.0242,0.2379,0.2621,NaN,3639.5494,3888.0370
83300,2013,DF,5300108,NaN,1302284,0.6339,35.5421,0.3661,0.0223,0.2341,0.2564,NaN,3782.9309,4097.1058


In [20]:
# rais["id_municipio"] = rais["id_municipio"].astype(int)

### df_ibge_população_estimada - 2002 - 2003 - 2004

In [21]:
df_IBGE_pop = pd.read_excel(r"C:\Users\DESKTOP\Desktop\Script -  Avaliação de impacto\dados_base's\IBGE_pop_municipios.xlsx", sheet_name='IBGE_pop_municipios')

df_IBGE_pop

,ANOS,SIGLA,U.F.,MUNIC,MUNICÍPIOS,POP
0,2002,GO,52,5,Abadia de Goiás,5415
1,2003,GO,52,5,Abadia de Goiás,5621
2,2004,GO,52,5,Abadia de Goiás,6054
3,2002,MG,31,10,Abadia dos Dourados,6436
4,2003,MG,31,10,Abadia dos Dourados,6432
...,...,...,...,...,...,...
16679,2003,MA,21,1400,Zé Doca,48565
16680,2004,MA,21,1400,Zé Doca,50183
16681,2002,SC,42,1985,Zortéa,2703
16682,2003,SC,42,1985,Zortéa,2736


.copy()

Tornando independente e tudo se altera sem mexer na variavél anterior

In [22]:
df_ibge = df_IBGE_pop.copy()

In [23]:
df_ibge.columns.tolist()

['ANOS', 'SIGLA', 'U.F.', 'MUNIC', 'MUNICÍPIOS', 'POP']

In [24]:
# # df_ibge.pd.to_numeric(df_ibge['COD.1'], errors='coerce')

# df_ibge['COD.1'].astype(float)

In [25]:
df_ibge.dtypes

ANOS           int64
SIGLA         object
U.F.           int64
MUNIC          int64
MUNICÍPIOS    object
POP            int64
dtype: object

In [26]:
# df_ibge = df_ibge.reset_index(drop=True)

In [27]:
df_ibge = df_ibge.rename(columns={
    "ANOS": "ano",
    "SIGLA": "uf",
    "POP": "População estimada"
})

In [28]:
df_ibge

,ano,uf,U.F.,MUNIC,MUNICÍPIOS,População estimada
0,2002,GO,52,5,Abadia de Goiás,5415
1,2003,GO,52,5,Abadia de Goiás,5621
2,2004,GO,52,5,Abadia de Goiás,6054
3,2002,MG,31,10,Abadia dos Dourados,6436
4,2003,MG,31,10,Abadia dos Dourados,6432
...,...,...,...,...,...,...
16679,2003,MA,21,1400,Zé Doca,48565
16680,2004,MA,21,1400,Zé Doca,50183
16681,2002,SC,42,1985,Zortéa,2703
16682,2003,SC,42,1985,Zortéa,2736


### (iii) estimar a mudança diferencial pós-PSI segundo a exposição inicial

3.1 Estratégia de identificação


"Utiliza-se o método de diferença em diferenças (DiD), que compara a variação do emprego e da massa salarial antes e depois de 2009 entre municípios com diferentes níveis de exposição ao Finame (Gertler et al., 2018; Cunningham, 2021). A mudança observada nos municípios menos expostos indica qual teria sido a mudança dos mais expostos sem a política, e a diferença entre as duas mudanças é atribuída ao programa (Cunningham, 2021; Gertler et al., 2018; Ferraro, 2009; Holland, 1985). Vale frisar que não é possível adotar uma variável dummy neste estudo, pois a intervenção atingiu todos os municípios na mesma data, julho de 2009. Por isso, segue-se Coleman e Feler (2014), que permitem que a intensidade do tratamento varie entre localidades. Assim, os municípios mais expostos são os mais afetados pelo choque, e os menos expostos fazem o papel de comparação.
O método DiD explora três características da intervenção: um choque comum e datado, que é a implantação do PSI em 2009; um painel com anos anteriores ao choque; e uma medida de exposição definida antes do programa, que varia entre municípios. O desenho é, portanto, um DiD com tratamento contínuo, no qual a dose é a exposição pré-determinada 𝑍 (Callaway; Goodman-Bacon; Sant'Anna, 2024)." (pág. 6)

### df_FINAME_pre_exposicao_2002_2004

In [29]:
df_finame_pre_exposure = df_finame.copy()

In [30]:
df_finame_pre_exposure = df_finame[
    (df_finame["ano"] >= 2002) &
    (df_finame["ano"] <= 2004)
].copy()

df_finame_pre_exposure = df_finame_pre_exposure.rename(
    columns={
        "municipio": "Municipios",
        "municipio_codigo": "id_municipio"
    }
)

print("Total de registros:", len(df_finame_pre_exposure))
print("Duplicações:", df_finame_pre_exposure.duplicated().sum())

df_finame_pre_exposure = df_finame_pre_exposure.reset_index()

Total de registros: 21391
Duplicações: 0


In [31]:
df_finame_pre_exposure

,index,ano,mes,forma_de_apoio,produto,instrumento_financeiro,inovacao,porte_de_empresa,regiao,uf,Municipios,id_municipio,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,desembolsos_reais,uf_sigla
0,23546,2002,1,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUDESTE,ESPIRITO SANTO,VITORIA,3205309,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,14264832.0000,ES
1,23547,2002,1,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,NORTE,PARA,ORIXIMINA,1505304,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,12246392.6700,PA
2,23548,2002,1,INDIRETA,BNDES FINAME,OUTROS,NﾃO,MﾉDIA,SUDESTE,SAO PAULO,SAO PAULO,3550308,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,4843518.3400,SP
3,23549,2002,1,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUDESTE,MINAS GERAIS,OURO BRANCO,3145901,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,4643467.8700,MG
4,23550,2002,1,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUDESTE,SAO PAULO,SAO PAULO,3550308,COMﾉRCIO E SERVIﾇOS,ELETRICIDADE E GﾁS,INFRAESTRUTURA,ENERGIA ELﾉTRICA,4093336.4000,SP
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21386,48338,2004,12,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUDESTE,SAO PAULO,SUMARE,3552403,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,360869.7300,SP
21387,48342,2004,12,INDIRETA,BNDES FINAME,OUTROS,NﾃO,MﾉDIA,SUL,RIO GRANDE DO SUL,CANOAS,4304606,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,348600.0000,RS
21388,48365,2004,12,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUL,SANTA CATARINA,LAGES,4209300,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,299237.5500,SC
21389,48409,2004,12,INDIRETA,BNDES FINAME,OUTROS,NﾃO,GRANDE,SUL,SANTA CATARINA,SAO LUDGERO,4217006,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BORRACHA E PLﾁSTICO,INDUSTRIA,OUTRAS,228000.0000,SC


### df_finame_pre_expo_popul_ibge

Aplicar merge entre o dataframe de pre exposição e a dataframe do IBGE população

In [32]:
df_ibge = df_ibge.copy()

In [33]:
df_ibge["municipio_merge"] = (
    df_ibge["MUNICÍPIOS"]
    .astype(str).apply(unidecode).str.upper().str.strip()
)

In [34]:
df_ibge = df_ibge.rename(columns={"municipio_merge": "Municipios"})

df_ibge

,ano,uf,U.F.,MUNIC,MUNICÍPIOS,População estimada,Municipios
0,2002,GO,52,5,Abadia de Goiás,5415,ABADIA DE GOIAS
1,2003,GO,52,5,Abadia de Goiás,5621,ABADIA DE GOIAS
2,2004,GO,52,5,Abadia de Goiás,6054,ABADIA DE GOIAS
3,2002,MG,31,10,Abadia dos Dourados,6436,ABADIA DOS DOURADOS
4,2003,MG,31,10,Abadia dos Dourados,6432,ABADIA DOS DOURADOS
...,...,...,...,...,...,...,...
16679,2003,MA,21,1400,Zé Doca,48565,ZE DOCA
16680,2004,MA,21,1400,Zé Doca,50183,ZE DOCA
16681,2002,SC,42,1985,Zortéa,2703,ZORTEA
16682,2003,SC,42,1985,Zortéa,2736,ZORTEA


In [35]:
df_finame_pre_exposure.columns.tolist()

['index',
 'ano',
 'mes',
 'forma_de_apoio',
 'produto',
 'instrumento_financeiro',
 'inovacao',
 'porte_de_empresa',
 'regiao',
 'uf',
 'Municipios',
 'id_municipio',
 'setor_cnae',
 'subsetor_cnae_agrupado',
 'setor_bndes',
 'subsetor_bndes',
 ' desembolsos_reais ',
 'uf_sigla']

In [36]:
df_finame_pre_exposure = df_finame_pre_exposure.rename(columns={"uf_sigla": "uf",
                                                                "uf": "Estado"  
                                                                })

In [37]:
df_ibge.columns.tolist()

['ano',
 'uf',
 'U.F.',
 'MUNIC',
 'MUNICÍPIOS',
 'População estimada',
 'Municipios']

In [38]:
df_merge_pop = pd.merge(
    df_finame_pre_exposure,
    df_ibge[["ano", "uf", "Municipios", "População estimada"]],
    on=["ano", "uf", "Municipios"],
    how="left",
    validate="many_to_one"
)

In [39]:
df_merge_pop.columns.tolist()

['index',
 'ano',
 'mes',
 'forma_de_apoio',
 'produto',
 'instrumento_financeiro',
 'inovacao',
 'porte_de_empresa',
 'regiao',
 'Estado',
 'Municipios',
 'id_municipio',
 'setor_cnae',
 'subsetor_cnae_agrupado',
 'setor_bndes',
 'subsetor_bndes',
 ' desembolsos_reais ',
 'uf',
 'População estimada']

In [40]:
# 1. Remover espaços dos nomes das colunas
df_merge_pop.columns = df_merge_pop.columns.str.strip()

# 2. Renomear as colunas
df_merge_pop = df_merge_pop.rename(columns={
    "desembolsos_reais": "R$_finame"
})

In [41]:
df_merge_pop.columns.tolist()

['index',
 'ano',
 'mes',
 'forma_de_apoio',
 'produto',
 'instrumento_financeiro',
 'inovacao',
 'porte_de_empresa',
 'regiao',
 'Estado',
 'Municipios',
 'id_municipio',
 'setor_cnae',
 'subsetor_cnae_agrupado',
 'setor_bndes',
 'subsetor_bndes',
 'R$_finame',
 'uf',
 'População estimada']

In [42]:
df_merge_pop = df_merge_pop[[
    'ano',
    'mes',
    'uf',
    'id_municipio',
    'Municipios',
    'População estimada',
    'porte_de_empresa',
    'setor_cnae',
    'subsetor_cnae_agrupado',
    'setor_bndes',
    'subsetor_bndes',
    'R$_finame'
]].copy()

df_merge_pop

,ano,mes,uf,id_municipio,Municipios,População estimada,porte_de_empresa,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,R$_finame
0,2002,1,ES,3205309,VITORIA,299357.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,14264832.0000
1,2002,1,PA,1505304,ORIXIMINA,49945.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,12246392.6700
2,2002,1,SP,3550308,SAO PAULO,10600060.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,4843518.3400
3,2002,1,MG,3145901,OURO BRANCO,31006.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,4643467.8700
4,2002,1,SP,3550308,SAO PAULO,10600060.0000,GRANDE,COMﾉRCIO E SERVIﾇOS,ELETRICIDADE E GﾁS,INFRAESTRUTURA,ENERGIA ELﾉTRICA,4093336.4000
...,...,...,...,...,...,...,...,...,...,...,...,...
21386,2004,12,SP,3552403,SUMARE,225307.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,360869.7300
21387,2004,12,RS,4304606,CANOAS,324994.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,348600.0000
21388,2004,12,SC,4209300,LAGES,165068.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,299237.5500
21389,2004,12,SC,4217006,SAO LUDGERO,9911.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BORRACHA E PLﾁSTICO,INDUSTRIA,OUTRAS,228000.0000


### Deflacionando - IPCA

Dezembro de 2008 é uma escolha metodologicamente interessante porque representa o encerramento do ano anterior à implementação do PSI, lançado em 2009.

Isso significa que podemos expressar todos os desembolsos em reais de dezembro de 2008, criando uma referência monetária comum anterior à intervenção.

Mês-base da deflação (dezembro de 2008): determina o poder de compra em que esses desembolsos serão expressos.

In [43]:
df_ipca = df_ipca.copy()

df_ipca

,Data,IPCA - Var. % mensal
0,2000-01-01,0.6200
1,2000-02-01,0.1300
2,2000-03-01,0.2200
3,2000-04-01,0.4200
4,2000-05-01,0.0100
...,...,...
175,2014-08-01,0.2500
176,2014-09-01,0.5700
177,2014-10-01,0.4200
178,2014-11-01,0.5100


.cumprod(), que significa produto acumulado.

multiplica progressivamente os fatores mensais para construir a trajetória do índice de preços.

* 100 apenas estabelece base 100

In [44]:
df_ipca["fator_mensal"] = (1 + df_ipca["IPCA - Var. % mensal"] / 100)

df_ipca["indice_ipca"] = (df_ipca["fator_mensal"].cumprod() * 100)

df_ipca

,Data,IPCA - Var. % mensal,fator_mensal,indice_ipca
0,2000-01-01,0.6200,1.0062,100.6200
1,2000-02-01,0.1300,1.0013,100.7508
2,2000-03-01,0.2200,1.0022,100.9725
3,2000-04-01,0.4200,1.0042,101.3965
4,2000-05-01,0.0100,1.0001,101.4067
...,...,...,...,...
175,2014-08-01,0.2500,1.0025,249.8270
176,2014-09-01,0.5700,1.0057,251.2510
177,2014-10-01,0.4200,1.0042,252.3063
178,2014-11-01,0.5100,1.0051,253.5930


Índice acumulado do IPCA de dezembro de 2008 : Índice de referência para atualizar os desembolsos do Finame.

entrega o filtro mas com a base inteira

In [45]:
ipca_base_2008 = df_ipca[df_ipca["Data"] == "2008-12-01"]

ipca_base_2008

,Data,IPCA - Var. % mensal,fator_mensal,indice_ipca
107,2008-12-01,0.2800,1.0028,182.1062


Valor fixo do indice ipca

In [46]:
ipca_base_2008 = df_ipca[df_ipca["Data"] == "2008-12-01"]["indice_ipca"].iloc[0]

ipca_base_2008

np.float64(182.10623562497008)

O deflacionamento consiste em retirar o efeito da inflação de um valor nominal (o preço de face da época) para transformá-lo em um valor real (em poder de compra atual ou de uma data-base específica).

O fator_deflacao é o número decimal pelo qual você divide o valor nominal para obter o valor corrigido.

Valor_Real = Valor_Nominal / Fator_Deflacao

In [47]:
df_ipca["fator_deflacao_dez_2008"] = (
    ipca_base_2008 / df_ipca["indice_ipca"]
)

# df_ipca.loc[df_ipca["ano"] == 2002]

df_ipca

,Data,IPCA - Var. % mensal,fator_mensal,indice_ipca,fator_deflacao_dez_2008
0,2000-01-01,0.6200,1.0062,100.6200,1.8098
1,2000-02-01,0.1300,1.0013,100.7508,1.8075
2,2000-03-01,0.2200,1.0022,100.9725,1.8035
3,2000-04-01,0.4200,1.0042,101.3965,1.7960
4,2000-05-01,0.0100,1.0001,101.4067,1.7958
...,...,...,...,...,...
175,2014-08-01,0.2500,1.0025,249.8270,0.7289
176,2014-09-01,0.5700,1.0057,251.2510,0.7248
177,2014-10-01,0.4200,1.0042,252.3063,0.7218
178,2014-11-01,0.5100,1.0051,253.5930,0.7181


In [48]:
df_ipca["ano"] = df_ipca["Data"].dt.year
df_ipca["mes"] = df_ipca["Data"].dt.month

df_ipca.columns.tolist()

['Data',
 'IPCA - Var. % mensal',
 'fator_mensal',
 'indice_ipca',
 'fator_deflacao_dez_2008',
 'ano',
 'mes']

In [49]:
df_ipca = df_ipca[[
 "ano",
 "mes",
 "Data",
 "IPCA - Var. % mensal",
 "fator_mensal",
 "indice_ipca",
 "fator_deflacao_dez_2008"    
]]

df_ipca

,ano,mes,Data,IPCA - Var. % mensal,fator_mensal,indice_ipca,fator_deflacao_dez_2008
0,2000,1,2000-01-01,0.6200,1.0062,100.6200,1.8098
1,2000,2,2000-02-01,0.1300,1.0013,100.7508,1.8075
2,2000,3,2000-03-01,0.2200,1.0022,100.9725,1.8035
3,2000,4,2000-04-01,0.4200,1.0042,101.3965,1.7960
4,2000,5,2000-05-01,0.0100,1.0001,101.4067,1.7958
...,...,...,...,...,...,...,...
175,2014,8,2014-08-01,0.2500,1.0025,249.8270,0.7289
176,2014,9,2014-09-01,0.5700,1.0057,251.2510,0.7248
177,2014,10,2014-10-01,0.4200,1.0042,252.3063,0.7218
178,2014,11,2014-11-01,0.5100,1.0051,253.5930,0.7181


### Deflacionando o df_merge_finame_pop

In [139]:
df_deflacionado = pd.merge(
    df_merge_pop,
    df_ipca[["ano", "mes", "fator_deflacao_dez_2008"]],
    on=["ano", "mes"],
    how="left",
    validate="many_to_one"
).copy()

df_deflacionado

,ano,mes,uf,id_municipio,Municipios,População estimada,porte_de_empresa,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,R$_finame,fator_deflacao_dez_2008
0,2002,1,ES,3205309,VITORIA,299357.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,14264832.0000,1.5877
1,2002,1,PA,1505304,ORIXIMINA,49945.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,12246392.6700,1.5877
2,2002,1,SP,3550308,SAO PAULO,10600060.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,4843518.3400,1.5877
3,2002,1,MG,3145901,OURO BRANCO,31006.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,4643467.8700,1.5877
4,2002,1,SP,3550308,SAO PAULO,10600060.0000,GRANDE,COMﾉRCIO E SERVIﾇOS,ELETRICIDADE E GﾁS,INFRAESTRUTURA,ENERGIA ELﾉTRICA,4093336.4000,1.5877
...,...,...,...,...,...,...,...,...,...,...,...,...,...
21386,2004,12,SP,3552403,SUMARE,225307.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,360869.7300,1.2059
21387,2004,12,RS,4304606,CANOAS,324994.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,348600.0000,1.2059
21388,2004,12,SC,4209300,LAGES,165068.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,299237.5500,1.2059
21389,2004,12,SC,4217006,SAO LUDGERO,9911.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BORRACHA E PLﾁSTICO,INDUSTRIA,OUTRAS,228000.0000,1.2059


In [140]:
df_deflacionado["R$_finame_deflacionado"] = df_deflacionado["R$_finame"] * df_deflacionado["fator_deflacao_dez_2008"]

df_deflacionado

,ano,mes,uf,id_municipio,Municipios,População estimada,porte_de_empresa,setor_cnae,subsetor_cnae_agrupado,setor_bndes,subsetor_bndes,R$_finame,fator_deflacao_dez_2008,R$_finame_deflacionado
0,2002,1,ES,3205309,VITORIA,299357.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,14264832.0000,1.5877,22648028.6574
1,2002,1,PA,1505304,ORIXIMINA,49945.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,12246392.6700,1.5877,19443387.2155
2,2002,1,SP,3550308,SAO PAULO,10600060.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,4843518.3400,1.5877,7689970.8435
3,2002,1,MG,3145901,OURO BRANCO,31006.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,MﾁQUINAS E EQUIPAMENTOS,INDUSTRIA,MECﾂNICA,4643467.8700,1.5877,7372354.1497
4,2002,1,SP,3550308,SAO PAULO,10600060.0000,GRANDE,COMﾉRCIO E SERVIﾇOS,ELETRICIDADE E GﾁS,INFRAESTRUTURA,ENERGIA ELﾉTRICA,4093336.4000,1.5877,6498919.8675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21386,2004,12,SP,3552403,SUMARE,225307.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,PRODUTOS ALIMENTﾍCIOS,INDUSTRIA,ALIMENTO E BEBIDA,360869.7300,1.2059,435173.2862
21387,2004,12,RS,4304606,CANOAS,324994.0000,MﾉDIA,COMﾉRCIO E SERVIﾇOS,TRANSPORTE TERRESTRE,INFRAESTRUTURA,TRANSPORTE RODOVIﾁRIO,348600.0000,1.2059,420377.2025
21388,2004,12,SC,4209300,LAGES,165068.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BEBIDAS,INDUSTRIA,ALIMENTO E BEBIDA,299237.5500,1.2059,360850.9586
21389,2004,12,SC,4217006,SAO LUDGERO,9911.0000,GRANDE,INDﾚSTRIA DE TRANSFORMAﾇﾃO,BORRACHA E PLﾁSTICO,INDUSTRIA,OUTRAS,228000.0000,1.2059,274945.5025


In [141]:
df_deflacionado.columns.tolist()

['ano',
 'mes',
 'uf',
 'id_municipio',
 'Municipios',
 'População estimada',
 'porte_de_empresa',
 'setor_cnae',
 'subsetor_cnae_agrupado',
 'setor_bndes',
 'subsetor_bndes',
 'R$_finame',
 'fator_deflacao_dez_2008',
 'R$_finame_deflacionado']

### Pre-exposição - [E_i]

- Somar os desembolsos anuais de cada município.
- Dividir pela população municipal do respectivo ano.
- Somar os valores per capita de 2002–2004 e dividir por três.

Somar o Finame por município e ano

In [ ]:
df_exposicao_Zi = df_deflacionado.groupby(["ano", 
    'id_municipio',
    "Municipios", 
    "População estimada"], as_index=False)["R$_finame_deflacionado"].sum()

df_sum_munc["R$_finame_deflacionado_per_capita"] = df_exposicao_Zi["R$_finame_deflacionado"] / df_sum_munc["População estimada"]

df_sum_munc["R$_finame_deflacionado_pre_exposicao_psi"]  = df_sum_munc.groupby(['id_municipio'])["R$_finame_deflacionado_per_capita"].transform("sum") / 3

df_sum_munc

,ano,id_municipio,Municipios,População estimada,R$_finame_deflacionado,R$_finame_deflacionado_per_capita,R$_finame_deflacionado_pre_exposicao_psi
0,2002,1100023,ARIQUEMES,78039.0000,1219609.3323,15.6282,12.3517
1,2002,1100056,CEREJEIRAS,17925.0000,1157934.4482,64.5989,38.4601
2,2002,1100072,CORUMBIARA,10160.0000,421214.2489,41.4581,76.7202
3,2002,1100114,JARU,54352.0000,386021.1091,7.1022,13.4128
4,2002,1100122,JI-PARANA,108694.0000,3402586.9315,31.3043,29.2993
...,...,...,...,...,...,...,...
4315,2004,5222005,VIANOPOLIS,12446.0000,1280027.3315,102.8465,285.5301
4316,2004,5222054,VICENTINOPOLIS,6415.0000,1788954.1449,278.8705,468.9844
4317,2004,5222203,VILA BOA,3516.0000,485129.7118,137.9777,152.8660
4318,2004,5222302,VILA PROPICIO,4845.0000,308524.0658,63.6789,103.9483


In [167]:
print("Finame:", len(df_finame_pre_exposure))
print("Após merge:", len(df_merge_pop))
print("Após IPCA:", len(df_deflacionado))
print("Após cálculo de exposição:", len(df_sum_munc))

Finame: 21391
Após merge: 21391
Após IPCA: 21391
Após cálculo de exposição: 4320


In [168]:
# df_sum_munc.to_excel(f"df_deflac_finame_psi_pre_exposicao_{data_hora}.xlsx", index=False)

Medida de pré exposição de acordo com a metodologia do trabalho

In [169]:
df_sum_munc.rename(columns={"R$_finame_deflacionado_pre_exposicao_psi": "pre_expo_E_i"}, inplace=True)

df_sum_munc.copy()

,ano,id_municipio,Municipios,População estimada,R$_finame_deflacionado,R$_finame_deflacionado_per_capita,pre_expo_E_i
0,2002,1100023,ARIQUEMES,78039.0000,1219609.3323,15.6282,12.3517
1,2002,1100056,CEREJEIRAS,17925.0000,1157934.4482,64.5989,38.4601
2,2002,1100072,CORUMBIARA,10160.0000,421214.2489,41.4581,76.7202
3,2002,1100114,JARU,54352.0000,386021.1091,7.1022,13.4128
4,2002,1100122,JI-PARANA,108694.0000,3402586.9315,31.3043,29.2993
...,...,...,...,...,...,...,...
4315,2004,5222005,VIANOPOLIS,12446.0000,1280027.3315,102.8465,285.5301
4316,2004,5222054,VICENTINOPOLIS,6415.0000,1788954.1449,278.8705,468.9844
4317,2004,5222203,VILA BOA,3516.0000,485129.7118,137.9777,152.8660
4318,2004,5222302,VILA PROPICIO,4845.0000,308524.0658,63.6789,103.9483


código para descobrir quantos municípios aparecem em 1, 2 ou 3 anos na sua base Finame (2002–2004).

.nunique()	Conta quantos anos diferentes existem para cada município


.value_counts()	Conta quantos municípios possuem 1, 2 ou 3 anos registrados


ajudando a identificar os anos sem registros de desembolsos.

In [170]:
df_sum_munc.groupby("Municipios")["ano"].nunique().value_counts()

ano
3    839
1    653
2    555
Name: count, dtype: int64

Verificando por municipio

In [171]:
df_sum_munc.loc[
    df_sum_munc["Municipios"] == "ABADIA DOS DOURADOS",
    ["ano", "R$_finame_deflacionado_per_capita",
     "pre_expo_E_i"]
]

,ano,R$_finame_deflacionado_per_capita,pre_expo_E_i
174,2002,69.4673,39.3522
1683,2003,48.5894,39.3522


In [172]:
df_sum_munc.columns.tolist()

['ano',
 'id_municipio',
 'Municipios',
 'População estimada',
 'R$_finame_deflacionado',
 'R$_finame_deflacionado_per_capita',
 'pre_expo_E_i']

### df_RAIS_merge

In [173]:
df_rais = df_rais.copy()
df_rais = df_rais[df_rais['ano'].isin([2002, 2003, 2004])]

df_rais = df_rais.reset_index(drop=True)

df_rais_filtro = df_rais

df_rais_filtro = df_rais_filtro[['ano',
 'sigla_uf',
 'id_municipio',
 'emprego_3112',
 'prop_homens',
 'prop_mulheres',
 'idade_media',
 'valor_remuneracao_media',
 'valor_remuneracao_dezembro']]

df_rais_filtro = df_rais_filtro.rename(columns={
     'sigla_uf': 'uf',
     'emprego_3112': 'estoque_emprego',
     'prop_homens': 'prop_homens',
     'prop_mulheres': 'prop_mulheres',
     'idade_media': 'idade_media',
     'valor_remuneracao_media': 'valor_remuneracao_media',
     'valor_remuneracao_dezembro': 'valor_remuneracao_dezembro'
})


df_rais_filtro = pd.merge(
    df_rais_filtro,
    df_sum_munc[[
        "ano",
        "id_municipio",
        "Municipios",
        "População estimada",
        "R$_finame_deflacionado",
        "R$_finame_deflacionado_per_capita",
        "pre_expo_E_i"
    ]],
    on=["id_municipio", "ano"],
    how="left",
    validate="one_to_one"
)


df_rais_filtro["R$_finame_deflacionado"] = (df_rais_filtro["R$_finame_deflacionado"].fillna(0))

df_rais_filtro["R$_finame_deflacionado_per_capita"] = (df_rais_filtro["R$_finame_deflacionado_per_capita"].fillna(0))

# df_rais_filtro["Municipios"] = (df_rais_filtro["Municipios"].fillna("0"))

# df_rais_filtro["População estimada"] = (df_rais_filtro["População estimada"].fillna("0"))

# Exposição histórica: mesma para todos os anos do município -- fixa por municipio
df_rais_filtro["pre_expo_E_i"] = (
    df_rais_filtro.groupby("id_municipio")["pre_expo_E_i"]
    .transform("max")
    .fillna(0)
)

# df_rais_filtro["expo_padroniz_Z_i"] = (df_rais_filtro["expo_padroniz_Z_i"].fillna("0"))


df_rais_filtro

,ano,uf,id_municipio,estoque_emprego,prop_homens,prop_mulheres,idade_media,valor_remuneracao_media,valor_remuneracao_dezembro,Municipios,População estimada,R$_finame_deflacionado,R$_finame_deflacionado_per_capita,pre_expo_E_i
0,2002,RO,1100015,1440,0.5785,0.4215,32.2160,438.4699,409.4442,NaN,NaN,0.0000,0.0000,0.0000
1,2003,RO,1100015,1629,0.5856,0.4144,32.4205,473.1530,503.8662,NaN,NaN,0.0000,0.0000,0.0000
2,2004,RO,1100015,1610,0.6050,0.3950,32.2075,520.5894,533.4424,NaN,NaN,0.0000,0.0000,0.0000
3,2002,RO,1100023,8543,0.6570,0.3430,31.2803,433.2104,453.8017,ARIQUEMES,78039.0000,1219609.3323,15.6282,12.3517
4,2003,RO,1100023,9391,0.6390,0.3610,31.4891,503.7372,585.6264,ARIQUEMES,79680.0000,1173418.7141,14.7266,12.3517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16651,2003,GO,5222302,346,0.6705,0.3295,37.3064,484.2254,609.7739,VILA PROPICIO,4704.0000,602805.4282,128.1474,103.9483
16652,2004,GO,5222302,228,0.6535,0.3465,36.9386,406.6344,409.8325,VILA PROPICIO,4845.0000,308524.0658,63.6789,103.9483
16653,2002,DF,5300108,813591,0.6678,0.3322,34.1030,1591.5898,1698.9732,BRASILIA,2145839.0000,66188344.4151,30.8450,23.5746
16654,2003,DF,5300108,810122,0.6631,0.3369,34.4100,1750.5040,1861.9699,BRASILIA,2189789.0000,60916168.8439,27.8183,23.5746


In [174]:
df_rais_filtro.columns.tolist()

['ano',
 'uf',
 'id_municipio',
 'estoque_emprego',
 'prop_homens',
 'prop_mulheres',
 'idade_media',
 'valor_remuneracao_media',
 'valor_remuneracao_dezembro',
 'Municipios',
 'População estimada',
 'R$_finame_deflacionado',
 'R$_finame_deflacionado_per_capita',
 'pre_expo_E_i']

### Pré-exposição padrão - Z_i

In [185]:
df_exposicao_Zi = df_rais_filtro[['ano',
 'uf',
 'id_municipio',
 'estoque_emprego',
 'prop_homens',
 'prop_mulheres',
 'idade_media',
 'valor_remuneracao_media',
 'valor_remuneracao_dezembro',
 'Municipios',
 'População estimada',
 'R$_finame_deflacionado',
 'R$_finame_deflacionado_per_capita',
 'pre_expo_E_i']].copy()

In [186]:
# Uma observação por município
# df_exposicao_Zi = df_rais_filtro[["id_municipio", "pre_expo_E_i"]].drop_duplicates(subset=["id_municipio"]).copy()

pre_expo_E_i = df_exposicao_Zi.drop_duplicates(subset=["id_municipio"])["pre_expo_E_i"]

In [187]:
pre_expo_E_i = df_exposicao_Zi["pre_expo_E_i"]

media_u_i = df_exposicao_Zi["pre_expo_E_i"].mean()

desv_pad_std = df_exposicao_Zi["pre_expo_E_i"].std()

In [188]:
df_exposicao_Zi["expo_padroniz_Z_i"] = (pre_expo_E_i - media_u_i) / desv_pad_std

df_exposicao_Zi["expo_padroniz_Z_i"] = pd.to_numeric(df_exposicao_Zi["expo_padroniz_Z_i"], errors="raise")

In [189]:
df_exposicao_Zi

,ano,uf,id_municipio,estoque_emprego,prop_homens,prop_mulheres,idade_media,valor_remuneracao_media,valor_remuneracao_dezembro,Municipios,População estimada,R$_finame_deflacionado,R$_finame_deflacionado_per_capita,pre_expo_E_i,expo_padroniz_Z_i
0,2002,RO,1100015,1440,0.5785,0.4215,32.2160,438.4699,409.4442,NaN,NaN,0.0000,0.0000,0.0000,-0.2538
1,2003,RO,1100015,1629,0.5856,0.4144,32.4205,473.1530,503.8662,NaN,NaN,0.0000,0.0000,0.0000,-0.2538
2,2004,RO,1100015,1610,0.6050,0.3950,32.2075,520.5894,533.4424,NaN,NaN,0.0000,0.0000,0.0000,-0.2538
3,2002,RO,1100023,8543,0.6570,0.3430,31.2803,433.2104,453.8017,ARIQUEMES,78039.0000,1219609.3323,15.6282,12.3517,-0.1834
4,2003,RO,1100023,9391,0.6390,0.3610,31.4891,503.7372,585.6264,ARIQUEMES,79680.0000,1173418.7141,14.7266,12.3517,-0.1834
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16651,2003,GO,5222302,346,0.6705,0.3295,37.3064,484.2254,609.7739,VILA PROPICIO,4704.0000,602805.4282,128.1474,103.9483,0.3386
16652,2004,GO,5222302,228,0.6535,0.3465,36.9386,406.6344,409.8325,VILA PROPICIO,4845.0000,308524.0658,63.6789,103.9483,0.3386
16653,2002,DF,5300108,813591,0.6678,0.3322,34.1030,1591.5898,1698.9732,BRASILIA,2145839.0000,66188344.4151,30.8450,23.5746,-0.1194
16654,2003,DF,5300108,810122,0.6631,0.3369,34.4100,1750.5040,1861.9699,BRASILIA,2189789.0000,60916168.8439,27.8183,23.5746,-0.1194


In [190]:
df_exposicao_Zi["expo_padroniz_Z_i"].describe()

count   16656.0000
mean        0.0000
std         1.0000
min        -0.2538
25%        -0.2538
50%        -0.2538
75%        -0.1253
max        25.4247
Name: expo_padroniz_Z_i, dtype: float64

In [191]:
print("Municípios:", df_exposicao_Zi["id_municipio"].nunique())

print("Município-ano duplicados:", df_exposicao_Zi.duplicated(["id_municipio", "ano"]).sum())

print("Exposições ausentes:", df_exposicao_Zi["pre_expo_E_i"].isna().sum())

print("Populações ausentes:", df_exposicao_Zi["População estimada"].isna().sum())

Municípios: 5558
Município-ano duplicados: 0
Exposições ausentes: 0
Populações ausentes: 12336


In [192]:
df_exposicao_Zi.drop_duplicates("id_municipio")["expo_padroniz_Z_i"].describe()

count   5558.0000
mean      -0.0003
std        0.9996
min       -0.2538
25%       -0.2538
50%       -0.2538
75%       -0.1254
max       25.4247
Name: expo_padroniz_Z_i, dtype: float64

### Regressão - Diferenças-em-Diferenças (DiD) com tratamento contínuo

### Salvando em excel

As variaveis criam salvam arquivos diferentes a cada dia

primerio tiramos o tempo --  data_hora

depois colocamos o caminho da variavel -- como base para ela sabe o que irá criar -- pode ser uma base qualquer

após isto -- .replace é substituir(".tipo", f_{função de hora})

por fim, salva .to_excel(arquivo_novo)

In [63]:
# # # # data_hora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# # # #--------------------------------------------------------------------------------------

# data_hora = datetime.now().strftime("%Y-%m-%d")

# SAVE = "df_pre_expo_2002_2004_finame_ibge_deflac_desagrupada.xlsx"

# SAVE_NEW = SAVE.replace(".xlsx", f"_{data_hora}.xlsx")

# df_deflacionado.to_excel(SAVE_NEW, index=False)

In [64]:
# data_hora = datetime.now().strftime("%Y-%m-%d")

# SAVE = "df_pre_expo_2002_2004_deflac_agrupad.xlsx".replace(".xlsx", f"_{data_hora}.xlsx")

# df_sum_munc.to_excel(SAVE, index=False)

Salvando -- .to_excel(arquivo, index=False)